# Capstone #3 — Production Customer Service Agent

*Notebook 26*

Put it all together. Combine sessions, guardrails, and human-in-the-loop into a full production customer service agent.

<br>
<br>

**Topics:**

- Sessions for multi-turn conversation memory

- Guardrails blocking off-topic requests (Lesson 22)

- Human approval for sensitive actions (Lesson 24)

- Prompt injection awareness (Lesson 23)

- Tracing — inspecting the run in the OpenAI dashboard (Lesson 25)

---

## 🔧 Setup

In [ ]:
from pathlib import Path
from dotenv import load_dotenv
load_dotenv(dotenv_path=Path("..") / ".env")

from pydantic import BaseModel
from agents import (
    Agent,
    GuardrailFunctionOutput,
    InputGuardrail,
    InputGuardrailTripwireTriggered,
    Runner,
    SQLiteSession,
    function_tool,
    input_guardrail,
)

# Notebook-specific imports
import json

MODEL = "gpt-5-mini"

print(f"✅ Setup complete: Using {MODEL}")

---

## 🏗️ System Architecture

This capstone combines sessions (Lesson 19), guardrails (Lesson 22), and human-in-the-loop approval (Lesson 24) into a single production pipeline:

**Pipeline:**
```
Customer message
        ↓
[Input Guardrail] — blocks non-support requests
        ↓
Support Agent (with SQLiteSession)
    ├── lookup_order          → auto-executes
    ├── search_faq            → auto-executes
    └── process_refund       → pauses for approval
        ↓
[Approval flow] → auto or manual based on amount
        ↓
Response (saved to session)
```
Every run is automatically recorded for later inspection in the OpenAI dashboard.

---

## 🛠️ Phase 1: Define the Tools

In [ ]:
# Simulated order database
ORDERS = {
    "ORD-001": {"item": "Wireless Headphones", "price": 149.99, "status": "delivered", "date": "2025-12-10"},
    "ORD-002": {"item": "Phone Case", "price": 19.99, "status": "delivered", "date": "2025-12-12"},
    "ORD-003": {"item": "Laptop Stand", "price": 49.99, "status": "in transit", "date": "2025-12-15"},
    "ORD-004": {"item": "USB-C Cable", "price": 12.99, "status": "delivered", "date": "2025-12-08"}
}

# Simulated FAQ database
FAQS = [
    {"q": "return policy", "a": "We accept returns on eligible delivered orders. Items must be in original condition. Contact us to confirm eligibility."},
    {"q": "shipping time", "a": "Standard shipping takes 5-7 business days. Express shipping takes 2 business days."},
    {"q": "warranty", "a": "All electronics come with a 1-year manufacturer warranty. Contact us to initiate a claim."},
    {"q": "payment", "a": "We accept Visa, Mastercard, Amex, and PayPal. Payment is charged at time of order."}
]


@function_tool
def lookup_order(order_id: str) -> str:
    """Look up the details and status of a customer order by order ID."""
    order = ORDERS.get(order_id.upper())
    if not order:
        return f"Order {order_id} not found. Please check the order ID and try again."
    return (
        f"Order {order_id}: {order['item']} — ${order['price']:.2f}\n"
        f"Status: {order['status']} | Date: {order['date']}"
    )


@function_tool
def search_faq(query: str) -> str:
    """Search the FAQ database for answers to common customer questions."""
    query_lower = query.lower()
    for faq in FAQS:
        if faq["q"] in query_lower or any(word in query_lower for word in faq["q"].split()):
            return faq["a"]
    return "I couldn't find a specific FAQ for that question. Please contact our support team directly."


# `needs_approval=True` tells the SDK to pause before executing this tool
# and populate `result.interruptions` (handled in the pipeline below).
# General rule: read-only tools auto-run; write actions that change money, records, or external systems pause for approval.
@function_tool(needs_approval=True)
def process_refund(order_id: str, amount: float, reason: str) -> str:
    """Process a refund for a customer order. Requires human approval before executing."""
    return f"✅ Refund of ${amount:.2f} processed for order {order_id}. Reason: {reason}"


# --------------------------------------------------------------
print("✅ Tools defined")
print("   lookup_order    — auto")
print("   search_faq      — auto")
print("   process_refund  — requires approval")

⚠️ **Security note:** The model chooses the refund `amount` and `reason`. In production, always validate the amount against the order record before approving — the approval gate checks permission, not correctness.

---

## 🛡️ Phase 2: Block Off-Topic Requests

A guardrail agent screens the message before it reaches the support agent — non-support requests get rejected up front. A keyword check would be brittle; a dedicated classifier agent handles typos, paraphrasing, and conversational language more reliably.

In [ ]:
class SupportTopicCheck(BaseModel):
    is_support_related: bool
    reasoning: str


topic_checker_instructions = (
    "Determine if the message is related to customer support for an online store.\n"
    "Support topics include: orders, refunds, returns, shipping, products, payments, warranties, and account issues.\n"
    "Return is_support_related=True only if it's clearly a customer support request."
)

topic_checker = Agent(
    name="TopicChecker",
    instructions=topic_checker_instructions,
    model=MODEL,
    output_type=SupportTopicCheck
)


@input_guardrail
async def support_topic_guardrail(
    ctx, agent: Agent, input
) -> GuardrailFunctionOutput:
    """Block requests unrelated to customer support."""

    result = await Runner.run(topic_checker, input, context=ctx.context)

    check = result.final_output
    return GuardrailFunctionOutput(
        tripwire_triggered=not check.is_support_related,
        output_info=check.reasoning
    )


# --------------------------------------------------------------
print("✅ Support topic guardrail ready")

---

⚠️ **Security note:** Customer messages are untrusted input and can contain prompt injection attacks. The input guardrail reduces the attack surface but cannot block injection inside support-related messages. Treat tool outputs and retrieved content as data, not instructions. (See Lesson 23)

---

## 🤖 Phase 3: Build the Agent

In [ ]:
support_instructions = (
    "You are a helpful customer support agent for an online electronics store.\n\n"
    "You can:\n"
    "- Look up order details and status with lookup_order\n"
    "- Answer questions about policies with search_faq\n"
    "- Process refunds with process_refund (for eligible delivered orders)\n\n"
    "Always be polite, empathetic, and solution-focused.\n"
    "If a customer asks for a refund, look up the order first to verify eligibility."
)

support_agent = Agent(
    name="CustomerSupport",
    instructions=support_instructions,
    model=MODEL,
    tools=[lookup_order, search_faq, process_refund],
    input_guardrails=[InputGuardrail(guardrail_function=support_topic_guardrail)]
)


# --------------------------------------------------------------
print("✅ Customer support agent ready")
print("   Tools: lookup_order, search_faq, process_refund")
print("   Guardrail: support topic filter")
print("   Session: will be passed at runtime")

## 🚀 Phase 4: The Full Pipeline

The orchestration layer handles sessions, guardrails, and the approval flow.

Focus on one idea here: `handle_customer_message()` is a plain Python wrapper around `Runner.run()` where you add app-specific rules — approval thresholds, logging, and fallback behavior.

- **Threshold routing** — refunds under $100 auto-approve; higher amounts pause for manual review

In [ ]:
AUTO_APPROVE_THRESHOLD = 100.0


async def handle_customer_message(message, session, auto_approve=True, agent=None):
    """Handle a customer message with full production safety features."""

    run_agent = agent or support_agent

    # -------------------------------------------------------
    # Component 1: Guardrail check + initial run
    # -------------------------------------------------------
    try:
        result = await Runner.run(run_agent, input=message, session=session)

    except InputGuardrailTripwireTriggered:
        return "I'm sorry, I can only help with customer support questions about orders, returns, and products."

    # -------------------------------------------------------
    # Component 2: Human-in-the-loop approval flow
    # -------------------------------------------------------
    # The SDK pauses execution and populates result.interruptions when it hits a tool marked needs_approval=True.
    # We convert the result to a mutable state, decide whether to approve or reject, then resume the run.
    while result.interruptions:
        state = result.to_state()

        for interruption in state.get_interruptions():
            tool_name = interruption.raw_item.name
            args = json.loads(interruption.raw_item.arguments)

            if tool_name == process_refund.__name__:
                amount = args.get("amount", 0)

                if auto_approve and amount <= AUTO_APPROVE_THRESHOLD:
                    print(f"   ✅ Auto-approved: ${amount:.2f} refund")
                    state.approve(interruption)
                else:
                    print(f"   ⚠️  Manual review: ${amount:.2f} refund for {args.get('order_id')}")
                    print(f"      Reason: {args.get('reason')}")
                    decision = input("      Approve? (yes/no): ").strip().lower()
                    if decision in ["yes", "y"]:
                        print("      ✅ Approved by human")
                        state.approve(interruption)
                    else:
                        print("      🚫 Rejected by human")
                        state.reject(interruption, message="Refund requires additional verification.")
            else:
                state.approve(interruption)

        result = await Runner.run(run_agent, state)

    return result.final_output


# --------------------------------------------------------------
print("✅ handle_customer_message() ready")
print(f"   Auto-approve threshold: ${AUTO_APPROVE_THRESHOLD}")

### 🔭 Tracing

Every `Runner.run()` call in this pipeline creates a trace automatically. Open `https://platform.openai.com/traces` and look for:
- The `TopicChecker` guardrail span — shows why an off-topic request was blocked before reaching the agent
- The `process_refund` interruption span — marks the exact pause point where the SDK stopped for approval
- The resumed run after approval — appears as a continuation of the same trace
- Token counts across turns — watch how multi-turn sessions grow

### 💡 Why This Works

`lookup_order` and `search_faq` are read-only tools — they auto-execute because they have no side effects. `process_refund` is a write action — it pauses for approval because it's irreversible. This is the read vs write safety principle from Lesson 23 applied in a production pipeline.

---

The $149.99 refund exceeds the $100 threshold — you'll see the approval prompt before the transaction completes.

⚠️ Turn 2 will pause for refund approval — ORD-001 is $149.99, above the $100 auto-approve threshold.

You'll see an `input()` prompt at Turn 2 — type `yes` to approve the refund.

Watch turn 2 — the customer doesn't repeat the order ID. The session is what lets the agent carry ORD-001 forward, so customers don't restate context every turn.

In [ ]:
# One session per customer — persists memory across turns
customer_session = SQLiteSession("customer_demo_001")

conversation = [
    "Hi, my order is ORD-001. I need help with my headphones.",
    "They stopped working after 3 days. Can I get a refund?",
    "What's your return policy?",
]

print("🎬 CUSTOMER SERVICE DEMO")
print("=" * 60)

for turn_number, message in enumerate(conversation, 1):
    print(f"\n👤 Customer (Turn {turn_number}): {message}")
    response = await handle_customer_message(message, customer_session)
    print(f"🤖 Agent: {response}")
    print("-" * 60)

#### Test: Off-Topic Request (Guardrail)

In [ ]:
print("\n🛡️  GUARDRAIL TEST")
print("=" * 60)
off_topic = "Can you help me write a Python script?"
print(f"👤 Customer: {off_topic}")

response = await handle_customer_message(off_topic, customer_session)

print(f"🤖 Agent: {response}")

#### Cleanup

In [ ]:
await customer_session.clear_session()
print("✅ Session cleared")

---

A useful extension pattern is to parameterize your orchestration function with `agent`, so you can swap in upgraded versions without rewriting the approval flow.

### Exercise 1: Extend the Production Agent

*Covers: agent extension — adding tools, guardrails, and evaluation*

Extend the agent with a new tool, a stricter guardrail, and an evaluation pass.

In [ ]:
# --------------------------------------------------------------
# 💪 Exercise 1: Extend the Production Agent
# --------------------------------------------------------------
# Objective: Extend the production agent with new capabilities.
#
# Phases 1-4 (tools, guardrail, agent, pipeline) are provided above.
# Your job: implement the two extensions below.

# -------------------------------------------------------
# Extension 1: check_shipping_status tool
# -------------------------------------------------------
# TODO 1: Create a @function_tool called check_shipping_status
#            - Parameter: order_id (str)
#            - Look up the order in ORDERS dict
#            - Return a shipping status message
#            - No approval needed

# -------------------------------------------------------
# Extension 2: Update the agent
# -------------------------------------------------------
# TODO 2: Create extended_support_agent — copy support_agent
#            but add check_shipping_status to the tools list
#            and update the instructions to mention it

# -------------------------------------------------------
# Extension 2b: Stricter guardrail
# -------------------------------------------------------
# TODO 2b: Update topic_checker_instructions or create an enhanced guardrail
#            that also rejects competitor comparison requests
#            (e.g. "How does your return policy compare to Amazon's?")
#            Wire the updated guardrail into extended_support_agent

# -------------------------------------------------------
# Extension 3: Test it
# -------------------------------------------------------
# TODO 3: Create a new SQLiteSession
#            Run handle_customer_message() but pass extended_support_agent
#            (you'll need to modify handle_customer_message or create a new version)
#            Test with: "Where is my order ORD-003?"
#            Print the response

# -------------------------------------------------------
# Extension 4: Evaluate the agent
# -------------------------------------------------------
# TODO 4: Build a small golden test set — 3 to 5 test cases covering:
#            - A valid order lookup
#            - A valid refund request
#            - An off-topic request that should be blocked
#           Use the judge agent pattern from Lesson 09 to score each response
#           against a rubric: correct tool used, guardrail behaved correctly,
#           response was helpful and accurate

print("💡 Implement the extensions above, then test them!")

---

## 🎯 Key Takeaways

**Production Features Work Together:**

- Sessions preserve conversation context across turns — no need to repeat it each turn
- Guardrails filter bad inputs before they reach the main agent
- Human-in-the-loop protects against irreversible actions

<br>
<br>

**Layered Architecture:**

- Each feature is independent — add or remove without breaking others
- The orchestration function (`handle_customer_message`) is the integration point

<br>
<br>

**Threshold-Based Automation:**

- Auto-approve small actions, escalate large ones
- This pattern balances speed with safety without manual review of everything

<br>
<br>

**Prompt Injection Awareness:**

- The input guardrail filters off-topic requests — it reduces the attack surface but does not block injection within a support-related message
- Treat tool outputs and retrieved content as data, not instructions
- Keep system instructions separate from customer-supplied content

---

## 📍 Next Step

**Notebook 27: MCP Fundamentals**  

Connect your agents to the growing ecosystem of MCP servers for filesystem access, web browsing, and more.

##### 🔧 [Troubleshooting Guide](https://github.com/barrettscott/openai-agents/blob/main/TROUBLESHOOTING.md#lesson-26-capstone-3--customer-service-agent)

---